In [3]:
import pandas as pd
import plotly.graph_objects as go
import os
from IPython.display import display # Dùng để in bảng dữ liệu đẹp mắt trong Jupyter

# ==========================================
# 1. HÀM ĐẾM SỐ TỪ TRÙNG NHAU (Word Overlap)
# ==========================================
def count_overlap(words_a, words_b):
    if pd.isna(words_a) or pd.isna(words_b):
        return 0
    # Cắt chuỗi, xóa khoảng trắng và đưa về tập hợp (set) để đếm
    set_a = set([w.strip().lower() for w in str(words_a).split(',')])
    set_b = set([w.strip().lower() for w in str(words_b).split(',')])
    return len(set_a.intersection(set_b))

# ==========================================
# 2. LOGIC TÍNH TOÁN VÀ VẼ HEATMAP TRỰC TIẾP
# ==========================================
def compare_all_topics_inline(truth_path, communities_dir):
    print("⏳ Đang tải dữ liệu Topic gốc (Subreddit Thực tế)...")
    if not os.path.exists(truth_path):
        print(f"❌ LỖI: Không tìm thấy file gốc tại: {truth_path}")
        return
        
    # Đọc file Subreddit
    df_truth = pd.read_csv(truth_path)
    subreddits = df_truth['Subreddit'].tolist()
    truth_keywords = df_truth['Top 20 Keywords (LDA)'].tolist()
    
    algorithms = ['leiden', 'louvain', 'spectral']
    all_best_matches = [] # Danh sách lưu kết quả để làm báo cáo

    for algo in algorithms:
        comm_path = os.path.join(communities_dir, f"community_topics_{algo}.csv")
        
        if not os.path.exists(comm_path):
            print(f"⚠️ Bỏ qua thuật toán {algo.upper()} do không tìm thấy file: {comm_path}")
            continue
            
        print(f"\n{'='*60}\n🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: {algo.upper()}\n{'='*60}")
        df_algo = pd.read_csv(comm_path)
        
        communities = df_algo['Community_ID'].tolist()
        algo_keywords = df_algo['Top 20 Keywords (LDA)'].tolist()
        
        # 2.1 TẠO MA TRẬN ĐIỂM SỐ (Rows: Subreddit, Cols: Community)
        matrix = []
        for sub_kw in truth_keywords:
            row = []
            for comm_kw in algo_keywords:
                # Tính điểm trùng khớp cho từng ô
                overlap_score = count_overlap(sub_kw, comm_kw)
                row.append(overlap_score)
            matrix.append(row)
            
        # 2.2 TÌM "BEST MATCH" (CỤM NÀO KHỚP NHẤT VỚI SUBREDDIT NÀO)
        for col_idx, comm_id in enumerate(communities):
            col_scores = [matrix[row_idx][col_idx] for row_idx in range(len(subreddits))]
            max_score = max(col_scores)
            
            # Chỉ ghi nhận nếu có trùng khớp (tránh các cụm điểm 0 hoàn toàn)
            if max_score > 0:
                best_sub_idx = col_scores.index(max_score)
                all_best_matches.append({
                    'Algorithm': algo.capitalize(),
                    'Cluster (ID)': comm_id,
                    'Subreddit (Best Match)': subreddits[best_sub_idx],
                    'Score (Matching words)': f"{max_score}/20",
                    'Keywords Cluster (Algorithm)': algo_keywords[col_idx],
                    'Keywords Subreddit (Reality)': truth_keywords[best_sub_idx]
                })

        # 2.3 VẼ BIỂU ĐỒ HEATMAP
        comm_labels = [f"Cluster {c}" for c in communities]
        
        fig = go.Figure(data=go.Heatmap(
            z=matrix,
            x=comm_labels,
            y=subreddits,
            colorscale='Blues', # Tông màu xanh dương
            text=matrix,        # Hiện số trên ô
            texttemplate="%{text}",
            hoverongaps=False
        ))
        
        fig.update_layout(
            title=f"Heatmap: Level matching topic - Algorithm {algo.capitalize()}",
            xaxis_title="Algorithm community (Community ID)",
            yaxis_title="Reality community (Subreddit)",
            width=900,
            height=max(400, len(subreddits) * 30) # Tự động dãn chiều cao theo số lượng subreddit
        )
        
        # In biểu đồ ra ngay bên dưới ô code Jupyter
        fig.show() 

    # ==========================================
    # 3. IN BẢNG BÁO CÁO TỔNG HỢP RA MÀN HÌNH
    # ==========================================
    if all_best_matches:
        print(f"\n{'='*60}\n📋 BẢNG TỔNG HỢP: CÁC CỤM KHỚP NHẤT VỚI THỰC TẾ\n{'='*60}")
        df_report = pd.DataFrame(all_best_matches)
        
        # Sắp xếp ưu tiên các cụm có độ trùng khớp cao lên đầu
        df_report = df_report.sort_values(by=['Algorithm', 'Score (Matching words)'], ascending=[True, False])
        
        # Hiển thị bảng dạng format HTML đẹp của Jupyter
        display(df_report)
        
        # Lưu file CSV dự phòng
        os.makedirs("outputs", exist_ok=True)
        report_path = "outputs/topic_matching_report.csv"
        df_report.to_csv(report_path, index=False)
        print(f"\n💾 (Đã tự động lưu bảng báo cáo này tại: {report_path})")

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN VÀ CHẠY
# ==========================================
# THAY ĐỔI ĐƯỜNG DẪN TỚI FILE CỦA BẠN NẾU CẦN
TRUTH_PATH = "../../outputs/LDA/subreddit_topics_ground_truth.csv" 
COMMUNITIES_DIR = "../../outputs/LDA/" 

# Gọi hàm thực thi
compare_all_topics_inline(TRUTH_PATH, COMMUNITIES_DIR)

⏳ Đang tải dữ liệu Topic gốc (Subreddit Thực tế)...

🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: LEIDEN



🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: LOUVAIN



🎯 ĐANG PHÂN TÍCH VÀ VẼ BIỂU ĐỒ CHO: SPECTRAL



📋 BẢNG TỔNG HỢP: CÁC CỤM KHỚP NHẤT VỚI THỰC TẾ


,Algorithm,Cluster (ID),Subreddit (Best Match),Score (Matching words),Keywords Cluster (Algorithm),Keywords Subreddit (Reality)
2,Leiden,1,Brawlstars,9/20,"post, new, got, use, deck, brawl, brawler, red...","post, subreddit, brawlstars, questions, remove..."
5,Leiden,3,Genshin_Impact,9/20,"xbox, switch, support, removed, new, need, iss...","characters, character, team, columbina, use, s..."
19,Leiden,19,Genshin_Impact,6/20,"ship, planet, freighter, save, base, need, spa...","characters, character, team, columbina, use, s..."
21,Leiden,21,skyrim,5/20,"skyrim, bugs, said, mods, mod, thats, dont, kn...","skyrim, know, use, armor, mod, mods, level, qu..."
3,Leiden,6,Destiny,20/20,"trump, right, know, shit, going, bad, did, act...","trump, right, know, shit, going, did, bad, cou..."
6,Leiden,4,Battlefield,20/20,"maps, battlefield, bf6, players, map, better, ...","maps, battlefield, bf6, players, map, playing,..."
8,Leiden,9,Minecraft,20/20,"post, comment, minecraft, quality, purpose, do...","post, comment, minecraft, quality, purpose, do..."
4,Leiden,7,deadbydaylight,19/20,"killer, survivor, survivors, seconds, killers,...","killer, survivor, survivors, seconds, killers,..."
7,Leiden,5,2007scape,19/20,"way, lol, content, know, got, account, did, ne...","way, content, lol, got, account, know, did, do..."
10,Leiden,12,ffxiv,19/20,"know, way, going, content, ll, new, ddos, need...","know, content, way, going, ll, ddos, new, need..."



💾 (Đã tự động lưu bảng báo cáo này tại: outputs/topic_matching_report.csv)
